# 01 · Exploración inicial de `jobs.csv`

Primer contacto con el dataset: qué columnas tiene, cuántos huecos hay, cuántas ofertas son
junior y cuántas de ellas piden 3 años o más de experiencia (la "inflación" del puesto junior).

Al final aparece un problema que no esperábamos: **muchas ofertas parecen repetidas**. Este
notebook lo detecta y lo mide; la causa y la regla de limpieza están en
`02_perfilado_calidad.ipynb`.

## Preparación

`RAW_DIR` es la ruta a `data/raw/`, definida en `src/radar_junior/config.py`, para no escribir
rutas a mano en cada celda.

In [16]:
import duckdb
from radar_junior.config import RAW_DIR

## 1. Primer vistazo a los ficheros

`DESCRIBE` muestra las columnas y el tipo que DuckDB deduce para cada una, sin cargar el
fichero entero. `LIMIT 5` enseña unas pocas filas para ver qué pinta tienen los datos.

In [17]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{RAW_DIR / 'open_jobs_daily.csv'}'").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,open_jobs,BIGINT,YES,None,None,None
2,fresher_friendly_jobs,BIGINT,YES,None,None,None


In [18]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{RAW_DIR/ 'jobs.csv'}'").df()

,column_name,column_type,null,key,default,extra
0,source,VARCHAR,YES,None,None,None
1,company,VARCHAR,YES,None,None,None
2,title,VARCHAR,YES,None,None,None
3,url,VARCHAR,YES,None,None,None
4,openings,BIGINT,YES,None,None,None
5,team,VARCHAR,YES,None,None,None
6,category,VARCHAR,YES,None,None,None
7,level,VARCHAR,YES,None,None,None
8,years_experience,BIGINT,YES,None,None,None
9,employment,VARCHAR,YES,None,None,None


In [19]:
duckdb.sql(f"SELECT * FROM '{RAW_DIR / 'jobs.csv'}'LIMIT 5").df()


,source,company,title,url,openings,team,category,level,years_experience,employment,...,skills,pay_min,pay_max,pay_currency,posted,first_seen,closed_at,ghost_score,ghost_reasons,fresher_friendly
0,adp,CITY Laundering,Route Representative,https://workforcenow.adp.com/mascsr/default/md...,1,None,other,mid,<NA>,full_time,...,None,<NA>,<NA>,None,2026-07-10,2026-07-13,2026-11-08,<NA>,None,False
1,adp,CITY Laundering,Route Service Apprentice,https://workforcenow.adp.com/mascsr/default/md...,1,None,other,junior,<NA>,full_time,...,None,<NA>,<NA>,None,2026-05-27,2026-05-30,2026-11-08,<NA>,None,True
2,adp,CITY Laundering,Route Service Representative,https://workforcenow.adp.com/mascsr/default/md...,1,None,other,mid,<NA>,full_time,...,None,<NA>,<NA>,None,2026-08-25,2026-08-26,2026-11-08,<NA>,None,False
3,adp,CITY Laundering,Textile Control Manager,https://workforcenow.adp.com/mascsr/default/md...,1,None,other,mid,<NA>,NaN,...,None,<NA>,<NA>,None,2026-08-28,2026-08-29,2026-11-08,<NA>,None,False
4,adp,CITY Laundering,Truck Loader/ Receiving Operator,https://workforcenow.adp.com/mascsr/default/md...,1,None,manufacturing,mid,<NA>,full_time,...,None,<NA>,<NA>,None,2026-07-10,2026-07-13,2026-11-08,<NA>,None,False


## 2. Perfil de `jobs.csv`

`duckdb.read_csv` crea la tabla `jobs` que usan el resto de consultas. `posted` se lee como
texto (`VARCHAR`) porque trae fechas rotas que harían fallar la lectura como fecha.

`SUMMARIZE` calcula, para cada columna, mínimo, máximo, valores distintos y porcentaje de nulos
de una sola vez.

In [20]:
jobs = duckdb.read_csv(RAW_DIR / "jobs.csv", dtype={"posted": "VARCHAR"})
duckdb.sql("SUMMARIZE SELECT * FROM jobs").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,source,VARCHAR,adp,zoho,84,NaN,NaN,NaN,NaN,NaN,11462654,0.00
1,company,VARCHAR,!K7 Music,🚀YOCHANA,385442,NaN,NaN,NaN,NaN,NaN,11462654,0.00
2,title,VARCHAR,! Berufskraftfahrer im nationalen Fernverkehr ...,🫵🏽SWITCHING CAREER? JOIN US SALES MARKETING RE...,3015815,NaN,NaN,NaN,NaN,NaN,11462654,0.00
3,url,VARCHAR,AMR.group,www.tracygross.com,9610191,NaN,NaN,NaN,NaN,NaN,11462654,7.48
4,openings,BIGINT,1,866,160,1.0385900158898629,0.8827542262413756,1,1,1,11462654,0.00
5,team,VARCHAR,EX-Executive,🧑🏽‍💻 Engineering,100490,NaN,NaN,NaN,NaN,NaN,11462654,32.82
6,category,VARCHAR,administrative,support,22,NaN,NaN,NaN,NaN,NaN,11462654,0.00
7,level,VARCHAR,intern,staff+,7,NaN,NaN,NaN,NaN,NaN,11462654,0.00
8,years_experience,BIGINT,0,20,21,4.5526055524436355,3.3274732239339864,2,4,6,11462654,98.01
9,employment,VARCHAR,contract,part_time,4,NaN,NaN,NaN,NaN,NaN,11462654,29.26


**Lectura: conclusiones de `jobs.csv`**

**Tamaño:** 11.462.654 ofertas y 23 columnas.

**Columnas que usar con confianza**
- `level` (7 valores, de `intern` a `staff+`) y `category` (22 sectores): sin nulos, sirven para agrupar.
- `fresher_friendly` y `remote`: sin nulos.
- `first_seen` como fecha de publicación, en lugar de `posted`.

**Columnas con muchos huecos**
- `years_experience`: 98 % nulos. La inflación del puesto junior solo se puede medir en ~230.000 ofertas.
- `pay_min` / `pay_max`: 97,7 % nulos. Quedan ~265.000 ofertas con salario, probablemente sesgadas hacia EE. UU.
- `skills`: 85,5 % nulos. Cuando aparece, viene separada por `;`.
- `closed_at`: 85,8 % nulos, pero no es un error: vacío = oferta todavía abierta.

**Problemas de calidad**
- `posted` viene muy sucia (años como `0026`, fechas en otros idiomas). La descarto.
- `pay_max` llega a 315 millones: hay valores extremos que habrá que revisar.
- `pay_currency` mezcla 59 monedas, con mayúsculas y minúsculas inconsistentes.

> **Corrección posterior (ver `02_perfilado_calidad.ipynb`):** "`closed_at` vacío = sigue abierta" solo vale para las fuentes que no son `shard`. La fuente `shard` (57 % de las filas) nunca registra cierres.

## 3. ¿Cuántas ofertas son junior?

Primero, cuántas ofertas hay de cada `level`. Después, cómo se cruza `level` con
`fresher_friendly` (la marca del propio dataset para ofertas aptas para recién titulados).

In [21]:
duckdb.sql("""
    SELECT level, count(*) AS ofertas
    FROM jobs
    GROUP BY level
    ORDER BY ofertas DESC
""")

levelVARCHAR,ofertasBIGINT
mid,10004558
junior,433834
senior,418492
manager+,219528
intern,137895
lead,133788
staff+,114559


In [22]:
duckdb.sql("""
    SELECT level, fresher_friendly, count(*) AS ofertas
    FROM jobs
    GROUP BY level, fresher_friendly
    ORDER BY ofertas DESC
""")

levelVARCHAR,fresher_friendlyBOOLEAN,ofertasBIGINT
mid,False,9975206
junior,True,430804
senior,False,418492
manager+,False,219528
intern,True,137830
lead,False,133788
staff+,False,114559
mid,True,29352
junior,False,3030
intern,False,65


**Lectura:**
- `mid` es el 87 % de las ofertas. Como el nivel se deduce del título, en la práctica `mid`
  significa "no se sabe": la mayoría de títulos no dicen el nivel.
- Junior (433.834) e intern (137.895) son pocas en comparación.
- `fresher_friendly` coincide casi siempre con `level` junior/intern, así que **no destapa
  juniors ocultos** entre los `mid` (solo 29.352).
- Hay **3.030 ofertas junior que no son `fresher_friendly`**. Son las sospechosas de inflación.

## 4. Inflación de experiencia

¿Qué experiencia piden esas ofertas junior que no son `fresher_friendly`?

In [23]:
duckdb.sql("""
    SELECT years_experience, count(*) AS ofertas
    FROM jobs
    WHERE level = 'junior' AND fresher_friendly = false
    GROUP BY years_experience
    ORDER BY years_experience
""")

years_experienceBIGINT,ofertasBIGINT
3,1355
4,526
5,539
6,134
7,76
8,93
9,12
10,59
11,3
12,15


Casi todas piden entre 3 y 5 años, pero hay un **pico raro en 18 años** (162 ofertas). Calculamos
la inflación con y sin ese pico. `FILTER (WHERE ...)` cuenta solo las filas que cumplen la
condición, sin necesidad de otra consulta.

In [24]:
duckdb.sql("""
    SELECT
        count(*)                                                AS juniors_con_dato,
        count(*) FILTER (WHERE years_experience >= 3)           AS piden_3_o_mas,
        round(100.0 * piden_3_o_mas / juniors_con_dato, 1)      AS pct_inflacion,

        -- la misma cuenta quitando el pico sospechoso de 18 años
        count(*) FILTER (WHERE years_experience <> 18)          AS juniors_sin_18,
        count(*) FILTER (WHERE years_experience >= 3
                           AND years_experience <> 18)          AS piden_3_o_mas_sin_18,
        round(100.0 * piden_3_o_mas_sin_18 / juniors_sin_18, 1) AS pct_inflacion_sin_18
    FROM jobs
    WHERE level = 'junior'
      AND years_experience IS NOT NULL
""").df()

,juniors_con_dato,piden_3_o_mas,pct_inflacion,juniors_sin_18,piden_3_o_mas_sin_18,pct_inflacion_sin_18
0,38280,3030,7.9,38118,2868,7.5


### ¿Qué es el pico de 18 años?

Miramos los títulos y los sectores de esas 162 ofertas.

In [25]:
duckdb.sql("""
    SELECT title, company, category, url
    FROM jobs
    WHERE level = 'junior'
      AND years_experience = 18
    LIMIT 20
""").df()

,title,company,category,url
0,Automotive Sales Associate,Chevroletofwayzata,sales,https://careers.hireology.com/chevroletofwayza...
1,Automotive Sales Associate,Chevroletofwayzata,sales,https://careers.hireology.com/chevroletofwayza...
2,Associate Teacher,Security Benefit,education,https://jobs.dayforcehcm.com/en-US/hcmportal/C...
3,Firebirds Retail Sales Associate - PT,BDA,sales,https://job-boards.greenhouse.io/bdainc/jobs/4...
4,Carpenter Apprentice,ISEC,other,https://jobs.dayforcehcm.com/en-US/isec/CANDID...
5,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/1db23fe5-4ec4-4b9...
6,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/1e4b574c-f925-455...
7,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/24059a15-15a9-452...
8,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/24abf1e0-84cb-422...
9,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/24c56b8d-d902-4d0...


In [26]:
duckdb.sql("""
    SELECT category, count(*) AS ofertas
    FROM jobs
    WHERE level = 'junior'
      AND years_experience = 18
    GROUP BY category
    ORDER BY ofertas DESC
""").df()

,category,ofertas
0,sales,84
1,operations,37
2,retail,16
3,people,10
4,other,4
5,logistics,4
6,manufacturing,4
7,support,1
8,education,1
9,healthcare,1


**Lectura: inflación de experiencia en ofertas junior**

**Resultado:** el **7,5 %** de las ofertas junior que indican experiencia piden 3 años o más
(2.868 de 38.118).

**Decisión: excluir `years_experience = 18`.**
Las 162 ofertas junior con ese valor son puestos de entrada (sales, operations y retail
suman 137) con títulos como *Sales Associate* u *Operations Associate*. Lo más probable
es que el extractor haya leído como experiencia el requisito de edad mínima
("must be 18 years or older"). Incluirlas apenas cambia el resultado (7,9 % frente a 7,5 %),
así que la conclusión es robusta.

**Pendiente:** posibles duplicados. Gopuff publica la misma *Operations Associate* muchas
veces con URLs distintas (¿una por ciudad?). Se investiga en la sección 5.

## 5. Duplicados: el caso Gopuff

Gopuff es una empresa de reparto rápido con pequeños almacenes en muchas ciudades. Traemos sus
ofertas de *Operations Associate* a un DataFrame de pandas con `.df()` para analizarlas con
Python. Son pocas filas, así que caben en memoria sin problema.

`.shape` devuelve `(filas, columnas)`.

In [27]:
gopuff = duckdb.sql("""
    SELECT *
    FROM jobs
    WHERE company = 'Gopuff'
      AND title = 'Operations Associate'
""").df()

gopuff.shape

(27, 23)

`nunique()` cuenta los valores distintos de cada columna: muestra qué cambia entre las 27 filas
y qué se repite.

In [28]:
gopuff.nunique()

source               1
company              1
title                1
url                 27
openings             2
team                 1
category             1
level                1
years_experience     1
employment           1
remote               1
location            26
country              1
skills               1
pay_min              0
pay_max              0
pay_currency         0
posted              23
first_seen           8
closed_at            3
ghost_score          3
ghost_reasons        5
fresher_friendly     1
dtype: int64

**Lectura:** todo es igual (título, categoría, nivel, skills...) salvo `url` (27 distintas),
`location` (26) y las fechas. Una de las ubicaciones se repite: `value_counts()` dice cuál.

In [29]:
gopuff["location"].value_counts()

location
Birmingham 1183, England        2
Cardiff, Wales                  1
Harrow, London, England         1
Salford, England                1
Bromley, England                1
Tottenham, London, England      1
Hackney, London, England        1
Greenwich, London, England      1
Bristol, England                1
Acton, London, England          1
Croydon, London, England        1
Wandsworth, London, England     1
Derby, England                  1
Coventry, England               1
Newcastle, England              1
Chessington, London, England    1
Wembley, London, England        1
Kentish Town, London            1
Swansea, Wales                  1
Manchester Central, England     1
Cambridge, England              1
Nottingham, England             1
Liverpool, England              1
Battersea, London, England      1
East London                     1
Balham, London, England         1
Name: count, dtype: int64

Comparamos las dos filas de esa ubicación. La línea filtra filas (`gopuff[condición]`, como un
`WHERE`) y luego elige columnas (`[[...]]`, como un `SELECT`).

In [31]:
gopuff[gopuff["location"] == "Birmingham 1183, England"][["url", "posted", "first_seen", "openings", "closed_at"]]

,url,posted,first_seen,openings,closed_at
11,https://jobs.lever.co/gopuff/68d07db6-ce85-41a...,2026-10-26,2026-10-26,1,2026-11-19
12,https://jobs.lever.co/gopuff/6a5839ad-885d-447...,2026-11-19,2026-11-19,1,NaT


**Lectura:**
- Las ofertas están en **26 ciudades distintas** del Reino Unido → son **plazas reales** en
  almacenes diferentes, no duplicados.
- Birmingham aparece dos veces, pero es una **republicación**: la primera oferta se cerró el
  19 nov, el mismo día que se abrió la segunda. Nunca hubo dos abiertas a la vez.

Regla candidata: considerar duplicado cuando coinciden `company + title + location`.

## 6. ¿Cuánto afectaría esa regla a todo el dataset?

`count(DISTINCT (company, title, location))` cuenta los tríos distintos: dos filas con los tres
valores iguales cuentan como una. La diferencia con `count(*)` es lo que eliminaría la regla.

In [32]:
duckdb.sql("""
    SELECT
        count(*) AS filas,
        count(DISTINCT (company, title, location)) AS combinaciones
    FROM jobs
""").df()

,filas,combinaciones
0,11462654,5789624


**Lectura:** la regla eliminaría **5,7 millones de filas, casi la mitad del dataset**. En Gopuff
solo sobraba 1 de 27. Hay que entender de dónde sale esa diferencia antes de aplicar la regla.

### ¿Cuáles son los grupos más grandes?

In [33]:
duckdb.sql("""
    SELECT company, title, location, count(*) AS filas
    FROM jobs
    GROUP BY company, title, location
    ORDER BY filas DESC
    LIMIT 20
""").df()

,company,title,location,filas
0,Round Room,Retail Sales Representative,TCC - Client Careers Site,291
1,Jacksons,Retail Cashier,ExtraMile,133
2,Bulk Barn,Cashier/Stock Clerk,Corporate Stores,132
3,Von Maur,DRY GOODS SALES ASSOCIATE,Dry Goods External,107
4,Rocky's Ace Hardware,Cashier,Client Careers Site,100
5,Loomis,Armed Driver Guard,Client Careers Site,98
6,Rocky's Ace Hardware,Sales Associate,Client Careers Site,98
7,TGJones,Sales Assistant,Client Careers Site,94
8,Von Maur,DRY GOODS KEYHOLDER,Dry Goods External,90
9,Von Maur,SEASONAL DRY GOODS SALES SUPPORT,Dry Goods External,85


**Lectura:** en los grupos más grandes, `location` no es una ciudad sino el nombre del portal de
empleo (*Client Careers Site*, *Corporate Stores*...). Son cadenas de tiendas que contratan lo
mismo en todos sus locales; la regla juntaría plazas reales.

Pero estos 20 grupos suman unas 1.800 filas, nada comparado con 5,7 millones: el exceso tiene
que estar repartido en muchos grupos pequeños.

### ¿Cómo se reparten los tamaños de grupo?

La consulta de dentro (una **subconsulta**) calcula el tamaño de cada grupo; la de fuera cuenta
cuántos grupos hay de cada tamaño.

In [34]:
duckdb.sql("""
    SELECT filas, count(*) AS grupos
    FROM (
        SELECT company, title, location, count(*) AS filas
        FROM jobs
        GROUP BY company, title, location
    )
    GROUP BY filas
    ORDER BY filas
    LIMIT 20
""").df()

,filas,grupos
0,1,4856829
1,2,210134
2,3,27697
3,4,14740
4,5,14475
5,6,27928
6,7,45383
7,8,111604
8,9,273723
9,10,170288


**Lectura:** hasta 5 filas los grupos bajan, como cabría esperar. Pero entre **6 y 11 filas hay
una joroba**, con pico en 9, y a partir de 12 casi desaparece. Esos grupos reúnen unos
5,25 de los 5,7 millones de filas sobrantes. Una forma tan regular no parece cosa de las
empresas, sino de cómo se construyó el dataset.

## Resumen y siguiente paso

| Tema | Resultado |
|---|---|
| Tamaño | 11.462.654 ofertas, 23 columnas |
| Nivel | `mid` = 87 % (en la práctica, desconocido); junior + intern ≈ 572.000 |
| Inflación junior | **7,5 %** de las ofertas junior con experiencia indicada piden 3+ años (2.868 de 38.118) |
| `years_experience = 18` | Es la edad mínima, no experiencia → se excluye |
| Gopuff | 26 ciudades = plazas reales; 1 republicación en Birmingham |
| Duplicados | La regla `company + title + location` quitaría casi la mitad de las filas, concentradas en grupos de 6 a 11 |

**Continúa en `02_perfilado_calidad.ipynb`:** allí se ve que la joroba viene de la fuente
`shard`, que guarda una copia de cada oferta cada 2 días. Se define la regla de limpieza
definitiva y se comprueba que la inflación del 7,5 % se mantiene después de limpiar.